# GroupDRO++ Notebook (Simple Start)

Notebook này là phiên bản **đơn giản hóa** để triển khai ý tưởng GroupDRO++ từ paper
**"Automated Domain Discovery from Multiple Sources to Improve Zero-Shot Generalization"**.

## Cấu trúc 5 phần (roadmap)
1. **Part 1 (đang làm): Setup + Config + khung Eval tối giản**
2. Part 2: Data pipeline từ nhiều source + split (train/val/test/zero-shot)
3. Part 3: Domain discovery module (prototype) + gán pseudo-domain
4. Part 4: Train GroupDRO++ loop + logging (loss/group-weights/metrics)
5. Part 5: Evaluation đầy đủ (ERM vs GroupDRO vs GroupDRO++) + báo cáo

> Mục tiêu của Part 1: giữ mọi thứ rõ ràng, chạy được, và dễ mở rộng dần theo từng phần.


## Part 1 — Setup môi trường, seed, config và eval schema

Ở phần này chúng ta chỉ dựng **khung xương chuẩn**:
- import package
- set random seed để reproducible
- khai báo config tập trung
- tạo schema cho eval metrics để các phần sau tái sử dụng


## (Tuỳ chọn) Cài package nếu môi trường chưa có\nNếu chạy ở môi trường sạch và thiếu `numpy/torch`, bật cell bên dưới để cài nhanh.\n

In [ ]:
# !pip -q install numpy torch\n

In [ ]:
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional

import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE}")


### Vì sao cần cell trên?
- `set_seed(...)` giúp kết quả ổn định hơn giữa các lần chạy.
- `DEVICE` giúp notebook tự chọn GPU nếu có.
- Cách tách setup sớm giúp các phần sau chỉ tập trung vào thuật toán.


In [ ]:
@dataclass
class ExperimentConfig:
    # Reproducibility
    seed: int = 42

    # Data / batching
    batch_size: int = 128
    num_workers: int = 2

    # Optimization
    lr: float = 1e-3
    weight_decay: float = 1e-4
    max_epochs: int = 10

    # GroupDRO++ placeholders (sẽ mở rộng ở Part 3-4)
    use_groupdropp: bool = True
    group_update_lr: float = 0.05
    num_discovered_domains: int = 4

    # Eval schema
    eval_splits: List[str] = None
    primary_metric: str = "worst_group_acc"

    def __post_init__(self):
        if self.eval_splits is None:
            self.eval_splits = ["train", "val", "test", "zero_shot"]


CFG = ExperimentConfig()
print(asdict(CFG))


In [ ]:
def init_eval_store(cfg: ExperimentConfig) -> Dict[str, Dict[str, Optional[float]]]:
    # Khởi tạo nơi lưu metrics theo split.
    # Part 4-5 sẽ ghi kết quả thật vào đây.
    metric_template = {
        "avg_acc": None,
        "worst_group_acc": None,
        "avg_loss": None,
    }
    return {split: dict(metric_template) for split in cfg.eval_splits}


eval_store = init_eval_store(CFG)
eval_store


## Kết quả Part 1
Bạn đã có một khung notebook sạch và dễ mở rộng:
- config tập trung (`ExperimentConfig`)
- schema eval thống nhất (`init_eval_store`)
- sẵn sàng nối sang Part 2 (data nhiều nguồn + zero-shot split)

Nếu bạn đồng ý, bước tiếp theo mình sẽ làm **Part 2** theo đúng cấu trúc notebook gốc nhưng giữ code ngắn gọn.


## Part 2 — Data pipeline nhiều nguồn + split zero-shot (bản đơn giản)

Ở Part 2, ta chưa vội dùng dataset thật mà dựng **khung xử lý data** trước:
1. Khai báo nhiều nguồn dữ liệu (multi-source).
2. Tách split `train/val/test` cho nguồn seen.
3. Giữ một số nguồn làm `zero_shot` (không dùng trong train).

> Cách này giúp kiểm tra logic pipeline trước khi cắm dữ liệu thật ở bước mở rộng.


In [ ]:
from dataclasses import dataclass
from typing import Tuple


@dataclass
class SourceSpec:
    name: str
    num_samples: int
    seen_in_training: bool = True


def build_default_sources() -> List[SourceSpec]:
    """
    Demo 4 nguồn; source_d là zero-shot.
    Sau này bạn chỉ cần map các nguồn thật vào format SourceSpec.
    """
    return [
        SourceSpec("source_a", 1200, True),
        SourceSpec("source_b", 1000, True),
        SourceSpec("source_c", 900, True),
        SourceSpec("source_d", 700, False),  # zero-shot domain
    ]


SOURCES = build_default_sources()
[(s.name, s.num_samples, s.seen_in_training) for s in SOURCES]


### Giải thích
- `SourceSpec` là schema tối giản cho mỗi nguồn dữ liệu.
- `seen_in_training=False` nghĩa là nguồn đó chỉ để đánh giá zero-shot.
- Khi chuyển sang dữ liệu thật, bạn chỉ cần thay `num_samples` + loader tương ứng.


In [ ]:
def split_indices(n: int, train_ratio: float = 0.7, val_ratio: float = 0.15, seed: int = 42) -> Tuple[List[int], List[int], List[int]]:
    assert 0 < train_ratio < 1
    assert 0 < val_ratio < 1
    assert train_ratio + val_ratio < 1

    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_idx = idx[:n_train].tolist()
    val_idx = idx[n_train:n_train + n_val].tolist()
    test_idx = idx[n_train + n_val:].tolist()
    return train_idx, val_idx, test_idx


def build_multi_source_splits(sources: List[SourceSpec], seed: int = 42) -> Dict[str, Dict[str, List[int]]]:
    """
    Output format:
    {
      source_name: {
        "train": [...], "val": [...], "test": [...], "zero_shot": [...]
      }
    }
    - Seen source: có train/val/test
    - Zero-shot source: chỉ có zero_shot
    """
    out = {}
    for s in sources:
        if s.seen_in_training:
            tr, va, te = split_indices(s.num_samples, seed=seed)
            out[s.name] = {"train": tr, "val": va, "test": te, "zero_shot": []}
        else:
            out[s.name] = {"train": [], "val": [], "test": [], "zero_shot": list(range(s.num_samples))}
    return out


source_splits = build_multi_source_splits(SOURCES, seed=CFG.seed)
{src: {k: len(v) for k, v in d.items()} for src, d in source_splits.items()}


In [ ]:
def summarize_global_splits(source_splits: Dict[str, Dict[str, List[int]]]) -> Dict[str, int]:
    summary = {"train": 0, "val": 0, "test": 0, "zero_shot": 0}
    for src in source_splits.values():
        for split in summary.keys():
            summary[split] += len(src[split])
    return summary


global_split_summary = summarize_global_splits(source_splits)
print("Global split summary:", global_split_summary)

# Đồng bộ sang eval_store nếu cần mở rộng theo split toàn cục.
for split in CFG.eval_splits:
    if split not in global_split_summary:
        global_split_summary[split] = 0

global_split_summary


## Kết quả Part 2
- Đã có khung multi-source rõ ràng (`SourceSpec`, `build_multi_source_splits`).
- Đã phân tách được nguồn seen và zero-shot theo đúng tinh thần zero-shot generalization.
- Đã có thống kê toàn cục để nối trực tiếp vào train/eval loop ở Part 3-4.

Nếu bạn muốn, bước tiếp theo mình sẽ làm **Part 3: prototype domain discovery** theo style ngắn gọn tương tự.


## Part 3 — Prototype domain discovery (GroupDRO++ core idea)

Mục tiêu Part 3: tạo **pseudo-domain tự động** từ dữ liệu seen source.

Bản đơn giản ở đây gồm 3 bước:
1. Gom sample từ các seen source.
2. Tạo feature demo (để mô phỏng embedding từ backbone).
3. Chạy k-means đơn giản để gán `pseudo_domain_id`.

> Ở Part 4, `pseudo_domain_id` này sẽ được dùng làm group cho GroupDRO++.


In [ ]:
def collect_seen_records(source_splits: Dict[str, Dict[str, List[int]]]) -> List[Dict]:
    """
    Mỗi record đại diện cho một sample thuộc source seen.
    Giữ split để sau này có thể dùng train-only hoặc train+val tùy lựa chọn.
    """
    records = []
    for source_name, split_map in source_splits.items():
        for split in ["train", "val", "test"]:
            for sample_idx in split_map[split]:
                records.append({
                    "source": source_name,
                    "split": split,
                    "sample_idx": sample_idx,
                })
    return records


seen_records = collect_seen_records(source_splits)
print("Seen records:", len(seen_records))
seen_records[:3]


### Giải thích
- `seen_records` là bảng trung gian (list các dict) để dễ nối sang feature extraction.
- Với dữ liệu thật, bạn thay phần feature demo bằng embedding từ model (ví dụ penultimate layer).


In [ ]:
def build_demo_features(records: List[Dict], feat_dim: int = 16, seed: int = 42) -> torch.Tensor:
    """
    Tạo embedding giả lập có source-specific shift để domain discovery nhìn thấy khác biệt.
    Shape: [N, feat_dim]
    """
    rng = np.random.default_rng(seed)

    # source-specific shifts để tạo cấu trúc domain rõ ràng hơn
    source_names = sorted({r["source"] for r in records})
    source_to_shift = {s: rng.normal(loc=i * 0.8, scale=0.1, size=(feat_dim,)) for i, s in enumerate(source_names)}

    feats = []
    for r in records:
        base = rng.normal(loc=0.0, scale=1.0, size=(feat_dim,))
        vec = base + source_to_shift[r["source"]]
        feats.append(vec)

    return torch.tensor(np.stack(feats), dtype=torch.float32)


def simple_kmeans(x: torch.Tensor, k: int, num_iters: int = 30, seed: int = 42):
    """
    K-means tối giản bằng PyTorch để tránh thêm dependency.
    Returns: (assignments [N], centers [k, d])
    """
    g = torch.Generator(device=x.device)
    g.manual_seed(seed)

    n = x.shape[0]
    init_ids = torch.randperm(n, generator=g)[:k]
    centers = x[init_ids].clone()

    for _ in range(num_iters):
        dist = torch.cdist(x, centers)  # [N, k]
        assign = dist.argmin(dim=1)

        new_centers = []
        for cid in range(k):
            mask = assign == cid
            if mask.any():
                new_centers.append(x[mask].mean(dim=0))
            else:
                # cluster rỗng -> random re-init
                rid = torch.randint(0, n, size=(1,), generator=g).item()
                new_centers.append(x[rid])
        centers = torch.stack(new_centers, dim=0)

    final_dist = torch.cdist(x, centers)
    final_assign = final_dist.argmin(dim=1)
    return final_assign, centers


features = build_demo_features(seen_records, feat_dim=16, seed=CFG.seed)
assignments, centers = simple_kmeans(features, k=CFG.num_discovered_domains, seed=CFG.seed)
print("Feature shape:", tuple(features.shape))
print("Centers shape:", tuple(centers.shape))


In [ ]:
def attach_pseudo_domains(records: List[Dict], assignments: torch.Tensor) -> List[Dict]:
    out = []
    for r, a in zip(records, assignments.tolist()):
        item = dict(r)
        item["pseudo_domain_id"] = int(a)
        out.append(item)
    return out


def summarize_discovered_domains(records_with_domains: List[Dict]) -> Dict[int, Dict[str, int]]:
    summary: Dict[int, Dict[str, int]] = {}
    for r in records_with_domains:
        d = r["pseudo_domain_id"]
        s = r["source"]
        if d not in summary:
            summary[d] = {}
        summary[d][s] = summary[d].get(s, 0) + 1
    return summary


records_with_domains = attach_pseudo_domains(seen_records, assignments)
domain_summary = summarize_discovered_domains(records_with_domains)
domain_summary


## Kết quả Part 3
- Đã có pipeline domain discovery prototype: `records -> features -> k-means -> pseudo_domain_id`.
- Đầu ra `records_with_domains` đã sẵn sàng để dùng làm group labels trong GroupDRO++ training ở Part 4.
- Khi chuyển sang dữ liệu thật, chỉ cần thay `build_demo_features(...)` bằng embedding từ model.


## Part 4 — Train loop GroupDRO++ (bản đơn giản)

Trong Part 4, mình làm một phiên bản ngắn gọn nhưng đủ ý:
1. Tạo nhãn classification demo từ feature.
2. Tách train/val theo `records_with_domains`.
3. Train model với objective GroupDRO trên `pseudo_domain_id`.
4. Log `avg_acc`, `worst_group_acc`, và `group_weights` theo epoch.


In [ ]:
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def build_demo_labels(features: torch.Tensor) -> torch.Tensor:
    """
    Label demo nhị phân để có task phân loại tối giản.
    Bạn sẽ thay bằng label thật của dataset trong bản mở rộng.
    """
    score = features[:, 0] + 0.3 * features[:, 1]
    return (score > score.median()).long()


labels = build_demo_labels(features)
print('Label distribution:', torch.bincount(labels).tolist())


In [ ]:
def make_split_tensors(records_with_domains: List[Dict], features: torch.Tensor, labels: torch.Tensor):
    train_ids, val_ids = [], []
    for i, r in enumerate(records_with_domains):
        if r['split'] == 'train':
            train_ids.append(i)
        elif r['split'] == 'val':
            val_ids.append(i)

    def pack(ids: List[int]):
        x = features[ids]
        y = labels[ids]
        g = torch.tensor([records_with_domains[i]['pseudo_domain_id'] for i in ids], dtype=torch.long)
        return x, y, g

    return pack(train_ids), pack(val_ids)


(train_x, train_y, train_g), (val_x, val_y, val_g) = make_split_tensors(records_with_domains, features, labels)
print('Train shape:', tuple(train_x.shape), 'Val shape:', tuple(val_x.shape))


In [ ]:
def make_loader(x: torch.Tensor, y: torch.Tensor, g: torch.Tensor, batch_size: int, shuffle: bool):
    ds = TensorDataset(x, y, g)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def evaluate_group_metrics(model: nn.Module, x: torch.Tensor, y: torch.Tensor, g: torch.Tensor) -> Dict[str, float]:
    model.eval()
    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(dim=1)
        avg_acc = (pred == y).float().mean().item()

        group_accs = []
        for gid in sorted(g.unique().tolist()):
            mask = g == gid
            acc = (pred[mask] == y[mask]).float().mean().item()
            group_accs.append(acc)

        worst_group_acc = min(group_accs) if group_accs else 0.0
    return {
        'avg_acc': avg_acc,
        'worst_group_acc': worst_group_acc,
    }


def train_groupdropp_simple(train_loader, val_tensors, num_groups: int, cfg: ExperimentConfig):
    in_dim = train_loader.dataset.tensors[0].shape[1]
    num_classes = int(train_loader.dataset.tensors[1].max().item()) + 1

    model = nn.Linear(in_dim, num_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    # q là trọng số GroupDRO cho mỗi discovered group
    q = torch.ones(num_groups, device=DEVICE) / num_groups

    history = []
    val_x, val_y, val_g = [t.to(DEVICE) for t in val_tensors]

    for epoch in range(cfg.max_epochs):
        model.train()
        for xb, yb, gb in train_loader:
            xb, yb, gb = xb.to(DEVICE), yb.to(DEVICE), gb.to(DEVICE)
            logits = model(xb)

            # loss theo từng group trong mini-batch
            per_group_loss = torch.zeros(num_groups, device=DEVICE)
            for gid in range(num_groups):
                mask = gb == gid
                if mask.any():
                    per_group_loss[gid] = nn.functional.cross_entropy(logits[mask], yb[mask])

            # GroupDRO objective
            loss = (q * per_group_loss).sum()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # cập nhật group weights q (exponentiated gradient, bản đơn giản)
            with torch.no_grad():
                q = q * torch.exp(cfg.group_update_lr * per_group_loss.detach())
                q = q / q.sum()

        val_metrics = evaluate_group_metrics(model, val_x, val_y, val_g)
        row = {
            'epoch': epoch,
            'val_avg_acc': val_metrics['avg_acc'],
            'val_worst_group_acc': val_metrics['worst_group_acc'],
            'group_weights': q.detach().cpu().tolist(),
        }
        history.append(row)

    return model, history


In [ ]:
num_groups = CFG.num_discovered_domains
train_loader = make_loader(train_x, train_y, train_g, batch_size=CFG.batch_size, shuffle=True)

model_gdropp, train_history = train_groupdropp_simple(
    train_loader=train_loader,
    val_tensors=(val_x, val_y, val_g),
    num_groups=num_groups,
    cfg=CFG,
)

train_history[-1]


In [ ]:
# Ghi vào eval_store để giữ cùng schema từ Part 1
final_val = train_history[-1]
eval_store['val']['avg_acc'] = final_val['val_avg_acc']
eval_store['val']['worst_group_acc'] = final_val['val_worst_group_acc']

print('Updated eval_store[val]:', eval_store['val'])
print('Final group weights:', final_val['group_weights'])


## Kết quả Part 4
- Đã có training loop GroupDRO++ tối giản chạy được end-to-end.
- Đã log được 2 metric chính: `avg_acc` và `worst_group_acc`.
- Đã theo dõi được biến thiên `group_weights` qua các epoch.

Nếu bạn đồng ý, mình sẽ làm **Part 5**: so sánh ERM vs GroupDRO vs GroupDRO++ và phần report kết quả.


## Part 5 — Evaluation cuối: ERM vs GroupDRO (source) vs GroupDRO++ (pseudo-domain)

Part cuối sẽ:
1. Train baseline ERM.
2. Train GroupDRO với group = `source_id` (seen sources).
3. So sánh với GroupDRO++ (group = `pseudo_domain_id`) từ Part 4.
4. In bảng tổng hợp metric để bạn nhìn nhanh.


In [ ]:
def build_source_group_ids(records_with_domains: List[Dict]) -> torch.Tensor:
    sources = sorted({r['source'] for r in records_with_domains})
    s2id = {s: i for i, s in enumerate(sources)}
    gids = [s2id[r['source']] for r in records_with_domains]
    return torch.tensor(gids, dtype=torch.long), s2id


source_group_ids, source_id_map = build_source_group_ids(records_with_domains)
print('Source->ID:', source_id_map)


In [ ]:
def train_erm_simple(train_loader, val_tensors, cfg: ExperimentConfig):
    in_dim = train_loader.dataset.tensors[0].shape[1]
    num_classes = int(train_loader.dataset.tensors[1].max().item()) + 1

    model = nn.Linear(in_dim, num_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    val_x, val_y, val_g = [t.to(DEVICE) for t in val_tensors]
    history = []

    for epoch in range(cfg.max_epochs):
        model.train()
        for xb, yb, _gb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = nn.functional.cross_entropy(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        val_metrics = evaluate_group_metrics(model, val_x, val_y, val_g)
        history.append({
            'epoch': epoch,
            'val_avg_acc': val_metrics['avg_acc'],
            'val_worst_group_acc': val_metrics['worst_group_acc'],
        })
    return model, history


def train_groupdro_simple(train_loader, val_tensors, num_groups: int, cfg: ExperimentConfig):
    in_dim = train_loader.dataset.tensors[0].shape[1]
    num_classes = int(train_loader.dataset.tensors[1].max().item()) + 1

    model = nn.Linear(in_dim, num_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    q = torch.ones(num_groups, device=DEVICE) / num_groups
    val_x, val_y, val_g = [t.to(DEVICE) for t in val_tensors]
    history = []

    for epoch in range(cfg.max_epochs):
        model.train()
        for xb, yb, gb in train_loader:
            xb, yb, gb = xb.to(DEVICE), yb.to(DEVICE), gb.to(DEVICE)
            logits = model(xb)

            per_group_loss = torch.zeros(num_groups, device=DEVICE)
            for gid in range(num_groups):
                mask = gb == gid
                if mask.any():
                    per_group_loss[gid] = nn.functional.cross_entropy(logits[mask], yb[mask])

            loss = (q * per_group_loss).sum()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                q = q * torch.exp(cfg.group_update_lr * per_group_loss.detach())
                q = q / q.sum()

        val_metrics = evaluate_group_metrics(model, val_x, val_y, val_g)
        history.append({
            'epoch': epoch,
            'val_avg_acc': val_metrics['avg_acc'],
            'val_worst_group_acc': val_metrics['worst_group_acc'],
            'group_weights': q.detach().cpu().tolist(),
        })

    return model, history


In [ ]:
# Dataloader cho ERM (group tensor chỉ để giữ format TensorDataset)
train_loader_erm = make_loader(train_x, train_y, train_g, batch_size=CFG.batch_size, shuffle=True)

# Dataloader cho GroupDRO-source
train_source_g = source_group_ids[[i for i, r in enumerate(records_with_domains) if r['split'] == 'train']]
val_source_g = source_group_ids[[i for i, r in enumerate(records_with_domains) if r['split'] == 'val']]
train_loader_gdro_source = make_loader(train_x, train_y, train_source_g, batch_size=CFG.batch_size, shuffle=True)

# 1) ERM
model_erm, hist_erm = train_erm_simple(
    train_loader=train_loader_erm,
    val_tensors=(val_x, val_y, val_g),
    cfg=CFG,
)

# 2) GroupDRO-source
model_gdro_source, hist_gdro_source = train_groupdro_simple(
    train_loader=train_loader_gdro_source,
    val_tensors=(val_x, val_y, val_source_g),
    num_groups=len(source_id_map),
    cfg=CFG,
)

# 3) GroupDRO++ (đã train ở Part 4): dùng train_history
hist_gdropp = train_history

summary_table = {
    'ERM': hist_erm[-1],
    'GroupDRO_source': hist_gdro_source[-1],
    'GroupDRO++_pseudo': hist_gdropp[-1],
}
summary_table


In [ ]:
# Giữ eval_store đúng schema float ở Part 1 (chỉ lưu kết quả GroupDRO++ hiện tại).\ncomparison_report = {\n    'avg_acc': {\n        'ERM': summary_table['ERM']['val_avg_acc'],\n        'GroupDRO_source': summary_table['GroupDRO_source']['val_avg_acc'],\n        'GroupDRO++_pseudo': summary_table['GroupDRO++_pseudo']['val_avg_acc'],\n    },\n    'worst_group_acc': {\n        'ERM': summary_table['ERM']['val_worst_group_acc'],\n        'GroupDRO_source': summary_table['GroupDRO_source']['val_worst_group_acc'],\n        'GroupDRO++_pseudo': summary_table['GroupDRO++_pseudo']['val_worst_group_acc'],\n    }\n}\n\nprint('Final comparison (val):')\nprint('avg_acc:', comparison_report['avg_acc'])\nprint('worst_group_acc:', comparison_report['worst_group_acc'])\n\n# eval_store vẫn giữ format cho một run hiện tại (GroupDRO++ ở Part 4).\neval_store['val']\n

## Checklist rà soát notebook
- [x] Có đủ 5 phần theo roadmap.
- [x] Config + eval schema thống nhất từ đầu đến cuối.
- [x] Multi-source split có tách rõ seen vs zero-shot.
- [x] Domain discovery sinh `pseudo_domain_id` để train GroupDRO++.
- [x] Có benchmark cuối (ERM / GroupDRO-source / GroupDRO++-pseudo).

> Với dữ liệu thật, bạn thay 2 chỗ chính: (1) phần feature demo, (2) phần label demo.
